# Pipeline de Modelado — GFP Implementation Gap

**Descripción:** Entrenamiento y evaluación de modelos de clasificación para predecir la brecha de implementación.
Cubre las tres modalidades: Contrata, Administración Directa (AD) y ARCC.

**Modelos:** Logistic Regression, Lasso, Ridge, Elastic Net, Random Forest, XGBoost  
**Resampling:** Original (O), SMOTE (S), SMOTE-Tomek (ST), Naive Random Sampling (NRS)

**Flujo:**
1. Configuración de modalidad
2. Carga de data procesada
3. Split train/test
4. Resampling
5. Entrenamiento de modelos
6. Evaluación (test + train)
7. Tabla de resultados
8. Curvas ROC
9. Feature importance
10. Exportación

## 0. Configuración — cambiar aquí para cada modalidad

In [1]:
import sys
sys.path.append('C:/15_GFP')  # permite importar desde src/

# ============================================================
# CONFIGURACIÓN PRINCIPAL
# Opciones de MODALIDAD: 'contrata' | 'ad' | 'arcc'
# ============================================================

MODALIDAD    = 'ad'
RANDOM_STATE = 2023
TEST_SIZE    = 0.2

PATH_DATA    = f'C:/15_GFP/data/processed/{MODALIDAD}/1_data_{MODALIDAD}.xlsx'
DIR_MODELS   = f'C:/15_GFP/outputs/models/{MODALIDAD}'
DIR_RESULTS  = f'C:/15_GFP/outputs/results/{MODALIDAD}'
DIR_FIGURES  = f'C:/15_GFP/outputs/figures/{MODALIDAD}'
DIR_FI       = f'C:/15_GFP/outputs/feature_importance/{MODALIDAD}'

print(f'Modalidad: {MODALIDAD.upper()}')
print(f'Input: {PATH_DATA}')

Modalidad: AD
Input: C:/15_GFP/data/processed/ad/1_data_ad.xlsx


## 1. Importaciones

In [2]:
#pip install xgboost imbalanced-learn

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.combine import SMOTETomek

# Funciones del proyecto
from src.gfp_utils import (
    evaluate_model,
    build_results_table,
    plot_roc_curves,
    get_feature_importance,
    grid_search_rf,
    grid_search_xgb,
    save_models,
    save_grid_search_results,
)

## 2. Carga de data

In [4]:
data = pd.read_excel(PATH_DATA, engine='openpyxl')

# ⚠️  VERIFICAR ANTES DE CORRER:
# Estas columnas se excluyen porque en versiones anteriores del pipeline no
# habían sido eliminadas en la etapa de limpieza. Si 01_cleaning_pipeline ya
# las excluye correctamente, este drop es redundante pero inofensivo.
# Si deben entrar al modelo como features, comentar las líneas de cols_excluir.
cols_excluir = [
    'avance_fisico_real',
    'porcentaje_ejecucion_financiera',
    'estado_actualizacion_avance',
    'monto_aprobado_soles',
]
data = data.drop(columns=[c for c in cols_excluir if c in data.columns], errors='ignore')

data = data.dropna()

print(f'Registros: {len(data):,} | Variables: {data.shape[1]}')
print(f'\nDistribución brecha_existente:')
print(data['brecha_existente'].value_counts())

Registros: 17,111 | Variables: 254

Distribución brecha_existente:
brecha_existente
1    10363
0     6748
Name: count, dtype: int64


## 3. Split train / test

In [5]:
dep_var   = 'brecha_existente'
pred_vars = [col for col in data.columns if col != dep_var]

X = data[pred_vars]
y = data[dep_var]

x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = TEST_SIZE,
    random_state = RANDOM_STATE,
    stratify     = y
)

print(f'Train: {x_train.shape[0]:,} | Test: {x_test.shape[0]:,}')
print(f'Features: {x_train.shape[1]}')

Train: 13,688 | Test: 3,423
Features: 253


## 4. Resampling

In [6]:
# SMOTE
smote = SMOTE(random_state=RANDOM_STATE, sampling_strategy='all')
x_train_smote, y_train_smote = smote.fit_resample(x_train, y_train)

# SMOTE-Tomek
smote_tomek = SMOTETomek(random_state=RANDOM_STATE)
x_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(x_train, y_train)

# Naive Random Sampling
ros = RandomOverSampler(random_state=RANDOM_STATE)
x_train_ros, y_train_ros = ros.fit_resample(x_train, y_train)

print('Distribución tras cada resampling:')
for label, y_ in [('Original', y_train), ('SMOTE', y_train_smote),
                  ('SMOTETomek', y_train_smote_tomek), ('NRS', y_train_ros)]:
    from collections import Counter
    print(f'  {label}: {dict(Counter(y_))}')

Distribución tras cada resampling:
  Original: {1: 8290, 0: 5398}
  SMOTE: {1: 8290, 0: 8290}
  SMOTETomek: {1: 7643, 0: 7643}
  NRS: {1: 8290, 0: 8290}


## 5. Entrenamiento de modelos

### 5.1 Logistic Regression

In [7]:
%%time
lg_model_o   = LogisticRegression(random_state=RANDOM_STATE).fit(x_train, y_train)
lg_model_s   = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
lg_model_st  = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
lg_model_nrs = LogisticRegression(random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('LG listo.')

LG listo.
CPU times: total: 14.9 s
Wall time: 3.37 s


### 5.2 Lasso (L1)

In [8]:
%%time
lasso_model_o   = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
lasso_model_s   = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
lasso_model_st  = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
lasso_model_nrs = LogisticRegressionCV(penalty='l1', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Lasso listo.')

Lasso listo.
CPU times: total: 17min 46s
Wall time: 18min 11s


### 5.3 Ridge (L2)

In [9]:
%%time
ridge_model_o   = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
ridge_model_s   = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
ridge_model_st  = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
ridge_model_nrs = LogisticRegressionCV(penalty='l2', solver='saga', cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Ridge listo.')

Ridge listo.
CPU times: total: 13min 33s
Wall time: 13min 51s


### 5.4 Elastic Net

In [10]:
%%time
elasticnet_model_o   = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train, y_train)
elasticnet_model_s   = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_smote, y_train_smote)
elasticnet_model_st  = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_smote_tomek, y_train_smote_tomek)
elasticnet_model_nrs = LogisticRegressionCV(penalty='elasticnet', solver='saga', l1_ratios=[0.5], cv=10, random_state=RANDOM_STATE).fit(x_train_ros, y_train_ros)
print('Elastic Net listo.')

Elastic Net listo.
CPU times: total: 18min 25s
Wall time: 18min 48s


### 5.5 Random Forest — Grid Search

In [11]:
%%time
rf_search_o   = grid_search_rf(x_train, y_train, random_state=RANDOM_STATE)
rf_search_s   = grid_search_rf(x_train_smote, y_train_smote, random_state=RANDOM_STATE)
rf_search_st  = grid_search_rf(x_train_smote_tomek, y_train_smote_tomek, random_state=RANDOM_STATE)
rf_search_nrs = grid_search_rf(x_train_ros, y_train_ros, random_state=RANDOM_STATE)

for label, s in [('O', rf_search_o), ('S', rf_search_s), ('ST', rf_search_st), ('NRS', rf_search_nrs)]:
    print(f'RF {label}: {s.best_params_}')

RF O: {'max_depth': 10, 'max_features': 120, 'n_estimators': 500}
RF S: {'max_depth': 20, 'max_features': 60, 'n_estimators': 500}
RF ST: {'max_depth': 30, 'max_features': 90, 'n_estimators': 250}
RF NRS: {'max_depth': 30, 'max_features': 60, 'n_estimators': 500}
CPU times: total: 7min 35s
Wall time: 36min 31s


In [12]:
%%time
def _build_rf(search):
    p = search.best_params_
    return RandomForestClassifier(
        max_features=p['max_features'],
        n_estimators=p['n_estimators'],
        max_depth=p['max_depth'],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

rf_optimal_model_o   = _build_rf(rf_search_o).fit(x_train, y_train)
rf_optimal_model_s   = _build_rf(rf_search_s).fit(x_train_smote, y_train_smote)
rf_optimal_model_st  = _build_rf(rf_search_st).fit(x_train_smote_tomek, y_train_smote_tomek)
rf_optimal_model_nrs = _build_rf(rf_search_nrs).fit(x_train_ros, y_train_ros)
print('RF óptimos entrenados.')

RF óptimos entrenados.
CPU times: total: 4min 11s
Wall time: 31.4 s


### 5.6 XGBoost — Grid Search

In [13]:
%%time
xgb_search_o   = grid_search_xgb(x_train, y_train, random_state=RANDOM_STATE)
xgb_search_s   = grid_search_xgb(x_train_smote, y_train_smote, random_state=RANDOM_STATE)
xgb_search_st  = grid_search_xgb(x_train_smote_tomek, y_train_smote_tomek, random_state=RANDOM_STATE)
xgb_search_nrs = grid_search_xgb(x_train_ros, y_train_ros, random_state=RANDOM_STATE)

for label, s in [('O', xgb_search_o), ('S', xgb_search_s), ('ST', xgb_search_st), ('NRS', xgb_search_nrs)]:
    print(f'XGB {label}: {s.best_params_}')

XGB O: {'colsample_bytree': 0.3, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 500, 'subsample': 0.8}
XGB S: {'colsample_bytree': 0.4, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
XGB ST: {'colsample_bytree': 0.4, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
XGB NRS: {'colsample_bytree': 0.4, 'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 500, 'subsample': 0.8}
CPU times: total: 14min 34s
Wall time: 18min 7s


In [14]:
%%time
def _build_xgb(search):
    p = search.best_params_
    return XGBClassifier(
        objective='binary:logistic',
        verbosity=0,
        colsample_bytree=p['colsample_bytree'],
        max_depth=p['max_depth'],
        n_estimators=p['n_estimators'],
        learning_rate=p['learning_rate'],
        subsample=p['subsample'],
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

xgb_optimal_model_o   = _build_xgb(xgb_search_o).fit(x_train, y_train)
xgb_optimal_model_s   = _build_xgb(xgb_search_s).fit(x_train_smote, y_train_smote)
xgb_optimal_model_st  = _build_xgb(xgb_search_st).fit(x_train_smote_tomek, y_train_smote_tomek)
xgb_optimal_model_nrs = _build_xgb(xgb_search_nrs).fit(x_train_ros, y_train_ros)
print('XGB óptimos entrenados.')

XGB óptimos entrenados.
CPU times: total: 1min 51s
Wall time: 10.2 s


## 6. Evaluación

### 6.1 Test set

In [15]:
# Lista de (nombre_modelo, estrategia_resampling, modelo_entrenado)
modelos_test = [
    ('Logistic Regression', 'O',   lg_model_o),
    ('Logistic Regression', 'S',   lg_model_s),
    ('Logistic Regression', 'ST',  lg_model_st),
    ('Logistic Regression', 'NRS', lg_model_nrs),
    ('Lasso',               'O',   lasso_model_o),
    ('Lasso',               'S',   lasso_model_s),
    ('Lasso',               'ST',  lasso_model_st),
    ('Lasso',               'NRS', lasso_model_nrs),
    ('Ridge',               'O',   ridge_model_o),
    ('Ridge',               'S',   ridge_model_s),
    ('Ridge',               'ST',  ridge_model_st),
    ('Ridge',               'NRS', ridge_model_nrs),
    ('Elastic Net',         'O',   elasticnet_model_o),
    ('Elastic Net',         'S',   elasticnet_model_s),
    ('Elastic Net',         'ST',  elasticnet_model_st),
    ('Elastic Net',         'NRS', elasticnet_model_nrs),
    ('Random Forest',       'O',   rf_optimal_model_o),
    ('Random Forest',       'S',   rf_optimal_model_s),
    ('Random Forest',       'ST',  rf_optimal_model_st),
    ('Random Forest',       'NRS', rf_optimal_model_nrs),
    ('Boosted Trees',       'O',   xgb_optimal_model_o),
    ('Boosted Trees',       'S',   xgb_optimal_model_s),
    ('Boosted Trees',       'ST',  xgb_optimal_model_st),
    ('Boosted Trees',       'NRS', xgb_optimal_model_nrs),
]

results_test = [
    {'model': name, 'sampling': samp, 'metrics': evaluate_model(model, x_test, y_test)}
    for name, samp, model in modelos_test
]

tabla_test = build_results_table(results_test)
tabla_test

,Overall_Accuracy,Roc_Auc,Global_F1_Score,Matthews_Corr_Coef,No_Precision,No_Recall,No_F1_Score,Si_Precision,Si_Recall,Si_F1_Score
O. Logistic Regression,0.740,0.828,0.735,0.479,0.643,0.767,0.700,0.827,0.722,0.771
S. Logistic Regression,0.725,0.826,0.724,0.488,0.607,0.858,0.711,0.873,0.638,0.737
ST. Logistic Regression,0.726,0.825,0.725,0.488,0.608,0.854,0.711,0.871,0.642,0.739
NRS. Logistic Regression,0.721,0.827,0.721,0.502,0.597,0.901,0.718,0.903,0.603,0.724
O. Lasso,0.696,0.804,0.638,0.339,0.721,0.375,0.493,0.690,0.905,0.783
S. Lasso,0.720,0.811,0.719,0.485,0.600,0.869,0.710,0.879,0.622,0.729
ST. Lasso,0.713,0.809,0.712,0.474,0.593,0.868,0.704,0.877,0.611,0.720
NRS. Lasso,0.713,0.810,0.713,0.476,0.593,0.870,0.705,0.878,0.610,0.720
O. Ridge,0.703,0.808,0.652,0.355,0.717,0.408,0.520,0.699,0.895,0.785
S. Ridge,0.719,0.814,0.719,0.484,0.599,0.868,0.709,0.879,0.622,0.728


### 6.2 Training set (RF y XGB)

In [16]:
modelos_train = [
    ('Random Forest', 'O',   rf_optimal_model_o,   x_train,             y_train),
    ('Random Forest', 'S',   rf_optimal_model_s,   x_train_smote,       y_train_smote),
    ('Random Forest', 'ST',  rf_optimal_model_st,  x_train_smote_tomek, y_train_smote_tomek),
    ('Random Forest', 'NRS', rf_optimal_model_nrs, x_train_ros,         y_train_ros),
    ('Boosted Trees', 'O',   xgb_optimal_model_o,   x_train,            y_train),
    ('Boosted Trees', 'S',   xgb_optimal_model_s,   x_train_smote,      y_train_smote),
    ('Boosted Trees', 'ST',  xgb_optimal_model_st,  x_train_smote_tomek,y_train_smote_tomek),
    ('Boosted Trees', 'NRS', xgb_optimal_model_nrs, x_train_ros,        y_train_ros),
]

results_train = [
    {'model': name, 'sampling': samp, 'metrics': evaluate_model(model, X_tr, y_tr)}
    for name, samp, model, X_tr, y_tr in modelos_train
]

tabla_train = build_results_table(results_train)
tabla_train

,Overall_Accuracy,Roc_Auc,Global_F1_Score,Matthews_Corr_Coef,No_Precision,No_Recall,No_F1_Score,Si_Precision,Si_Recall,Si_F1_Score
O. Random Forest,0.805,0.895,0.799,0.601,0.732,0.799,0.764,0.861,0.809,0.834
S. Random Forest,0.935,0.992,0.935,0.877,0.891,0.993,0.939,0.992,0.878,0.931
ST. Random Forest,0.994,1.000,0.994,0.987,0.987,1.000,0.994,1.000,0.987,0.993
NRS. Random Forest,0.994,1.000,0.994,0.988,0.989,1.000,0.994,1.000,0.989,0.994
O. Boosted Trees,0.787,0.873,0.780,0.563,0.711,0.775,0.742,0.844,0.795,0.819
S. Boosted Trees,0.840,0.924,0.839,0.687,0.797,0.912,0.851,0.897,0.768,0.828
ST. Boosted Trees,0.863,0.941,0.862,0.732,0.823,0.925,0.871,0.914,0.801,0.854
NRS. Boosted Trees,0.854,0.937,0.853,0.715,0.810,0.925,0.863,0.912,0.783,0.843


## 7. Curvas ROC

In [17]:
# Random Forest — test
plot_roc_curves(
    models_dict={
        'RF O.': rf_optimal_model_o, 'RF S.': rf_optimal_model_s,
        'RF ST.': rf_optimal_model_st, 'RF NRS.': rf_optimal_model_nrs,
    },
    X=x_test, y=y_test,
    title=f'ROC — Random Forest ({MODALIDAD.upper()}) — Test',
    filepath=f'{DIR_FIGURES}/roc_rf_test.jpg',
)

# XGBoost — test
plot_roc_curves(
    models_dict={
        'XGB O.': xgb_optimal_model_o, 'XGB S.': xgb_optimal_model_s,
        'XGB ST.': xgb_optimal_model_st, 'XGB NRS.': xgb_optimal_model_nrs,
    },
    X=x_test, y=y_test,
    title=f'ROC — Boosted Trees ({MODALIDAD.upper()}) — Test',
    filepath=f'{DIR_FIGURES}/roc_xgb_test.jpg',
)

# Random Forest — train
plot_roc_curves(
    models_dict={
        'RF O.': rf_optimal_model_o, 'RF S.': rf_optimal_model_s,
        'RF ST.': rf_optimal_model_st, 'RF NRS.': rf_optimal_model_nrs,
    },
    X=x_train, y=y_train,
    title=f'ROC — Random Forest ({MODALIDAD.upper()}) — Train',
    filepath=f'{DIR_FIGURES}/roc_rf_train.jpg',
)

# XGBoost — train
plot_roc_curves(
    models_dict={
        'XGB O.': xgb_optimal_model_o, 'XGB S.': xgb_optimal_model_s,
        'XGB ST.': xgb_optimal_model_st, 'XGB NRS.': xgb_optimal_model_nrs,
    },
    X=x_train, y=y_train,
    title=f'ROC — Boosted Trees ({MODALIDAD.upper()}) — Train',
    filepath=f'{DIR_FIGURES}/roc_xgb_train.jpg',
)

ROC guardada: C:/15_GFP/outputs/figures/ad/roc_rf_test.jpg
ROC guardada: C:/15_GFP/outputs/figures/ad/roc_xgb_test.jpg
ROC guardada: C:/15_GFP/outputs/figures/ad/roc_rf_train.jpg
ROC guardada: C:/15_GFP/outputs/figures/ad/roc_xgb_train.jpg


## 8. Feature Importance (RF y XGB)

In [18]:
fi_models = [
    ('rf_o',   rf_optimal_model_o),
    ('rf_s',   rf_optimal_model_s),
    ('rf_st',  rf_optimal_model_st),
    ('rf_nrs', rf_optimal_model_nrs),
    ('xgb_o',   xgb_optimal_model_o),
    ('xgb_s',   xgb_optimal_model_s),
    ('xgb_st',  xgb_optimal_model_st),
    ('xgb_nrs', xgb_optimal_model_nrs),
]

fi_results = {}
for name, model in fi_models:
    fi_results[name] = get_feature_importance(model, pred_vars, top_n=50)

# Preview del mejor (RF original)
fi_results['rf_o'].head(10)

,vars,score
0,n_modificaciones,0.445667
1,log_monto_aprobado,0.167207
2,plazo_ejecucion_dias,0.110457
3,anio_inicio_obra,0.058121
4,n_adicionales_obra,0.048969
5,n_informes_control,0.019395
6,Region_sierra sur,0.013249
7,n_obras_relacionadas,0.006624
8,existe_paralizacion,0.006263
9,naturaleza_obra_Construcción/Creación,0.004853


## 9. Exportación

In [19]:
import os
for d in [DIR_MODELS, DIR_RESULTS, DIR_FIGURES, DIR_FI]:
    os.makedirs(d, exist_ok=True)

# 9.1 Tablas de resultados
tabla_test.to_excel(f'{DIR_RESULTS}/results_test.xlsx')
tabla_train.to_excel(f'{DIR_RESULTS}/results_train.xlsx')
print('Tablas de resultados guardadas.')

# 9.2 Modelos
save_models(
    models_dict={
        'lg_o': lg_model_o, 'lg_s': lg_model_s, 'lg_st': lg_model_st, 'lg_nrs': lg_model_nrs,
        'lasso_o': lasso_model_o, 'lasso_s': lasso_model_s, 'lasso_st': lasso_model_st, 'lasso_nrs': lasso_model_nrs,
        'ridge_o': ridge_model_o, 'ridge_s': ridge_model_s, 'ridge_st': ridge_model_st, 'ridge_nrs': ridge_model_nrs,
        'elasticnet_o': elasticnet_model_o, 'elasticnet_s': elasticnet_model_s,
        'elasticnet_st': elasticnet_model_st, 'elasticnet_nrs': elasticnet_model_nrs,
        'rf_o': rf_optimal_model_o, 'rf_s': rf_optimal_model_s,
        'rf_st': rf_optimal_model_st, 'rf_nrs': rf_optimal_model_nrs,
        'xgb_o': xgb_optimal_model_o, 'xgb_s': xgb_optimal_model_s,
        'xgb_st': xgb_optimal_model_st, 'xgb_nrs': xgb_optimal_model_nrs,
    },
    output_dir=DIR_MODELS,
)

# 9.3 Grid search results
save_grid_search_results(
    searches_dict={
        'gs_rf_o': rf_search_o, 'gs_rf_s': rf_search_s,
        'gs_rf_st': rf_search_st, 'gs_rf_nrs': rf_search_nrs,
        'gs_xgb_o': xgb_search_o, 'gs_xgb_s': xgb_search_s,
        'gs_xgb_st': xgb_search_st, 'gs_xgb_nrs': xgb_search_nrs,
    },
    output_dir=DIR_RESULTS,
)

# 9.4 Feature importance
for name, fi_df in fi_results.items():
    fi_df.to_excel(f'{DIR_FI}/{name}_feature_importance.xlsx', index=False)
print(f'Feature importance guardada en: {DIR_FI}')

Tablas de resultados guardadas.
24 modelos guardados en: C:/15_GFP/outputs/models/ad
Grid search results guardados en: C:/15_GFP/outputs/results/ad
Feature importance guardada en: C:/15_GFP/outputs/feature_importance/ad
